<!-- # The Bayesian Finite Element Method in Inverse Problems: Pullout Test

This notebook is associated with section 3.1 of "The Bayesian Finite Element Method in Inverse Problems: a Critical Comparison between Probabilistic Models for Discretization Error" by Anne Poot, Iuri Rocha, Pierre Kerfriden and Frans van der Meer ([doi:10.48550/arXiv.2506.02815](https://doi.org/10.48550/arXiv.2506.02815)). -->
# Convergence

In [ ]:
# general imports
import os
import numpy as np
from scipy.sparse.linalg import spsolve
import matplotlib.pyplot as plt

## Convergence in 1D

In [ ]:
# true solution
domain = np.linspace(0, 1, 1001)


def true_solution(x):
    return (1 - x**2) * (np.exp(2 * x) - 1)


def true_strain(x):
    return 2 * x + np.exp(2 * x) * (2 - 2 * x - 2 * x**2)


def true_source(x):
    return -2 - np.exp(2 * x) * (2 - 8 * x - 4 * x**2)


def true_avg_solution(a, b):
    # (1 - x**2) * (np.exp(2 * x) - 1)
    # -1            -> -x
    # x^2           -> 1/3 x^3
    # exp(2x)       -> 1/2 exp(2x)
    # -x^2 exp(2x)  -> (-1/2 x^2 + 1/2 x - 1/4) exp(2x)

    # total -> 1/3 x^3 - x + (1/4 + 1/2 x - 1/2 x^2) exp(2x)
    eval_a = a**3 / 3 - a + (0.25 + 0.5 * a - 0.5 * a**2) * np.exp(2 * a)
    eval_b = b**3 / 3 - b + (0.25 + 0.5 * b - 0.5 * b**2) * np.exp(2 * b)
    return (eval_b - eval_a) / (b - a)


u = np.zeros_like(domain)
eps = np.zeros_like(domain)
f = np.zeros_like(domain)

for i, x in enumerate(domain):
    u[i] = true_solution(x)
    eps[i] = true_strain(x)
    f[i] = true_source(x)

fig, axs = plt.subplots(ncols=3, figsize=(9, 3))
axs[0].plot(domain, u)
axs[1].plot(domain, eps)
axs[2].plot(domain, f)
plt.show()

In [ ]:
from scipy.optimize import minimize_scalar, root_scalar

u_argmax = minimize_scalar(lambda x: -true_solution(x), bounds=(0.0, 1.0)).x
eps_root = root_scalar(true_strain, bracket=(0.0, 1.0)).root
eps_argmax = minimize_scalar(lambda x: -true_strain(x), bounds=(0.0, 1.0)).x
f_root = root_scalar(true_source, bracket=(0.0, 1.0)).root

In [ ]:
print(np.isclose(u_argmax, eps_root))
print(np.isclose(eps_argmax, f_root))

In [ ]:
from fem.jive import CJiveRunner
from fem.meshing import mesh_interval_with_line2
from experiments.reproduction.theory.props import get_fem_props

In [ ]:
nodes, elems = mesh_interval_with_line2(n=8)
props = get_fem_props()
jive = CJiveRunner(props, elems=elems)
globdat = jive()

In [ ]:
x_h = nodes.get_coords().flatten()
u_h = globdat["state0"]

fig, ax = plt.subplots()
ax.plot(x_h, u_h)
plt.show()

In [ ]:
obs_points = np.linspace(0, 1, 16, endpoint=False)

In [ ]:
obs_points += 0.5 * (obs_points[1] - obs_points[0])
obs_radius = 0.01

### 1D prior convergence with reference mesh refinement

In [ ]:
def true_prior_source(x, a, b):
    if x < a:
        return 0
    elif x > b:
        return 0
    else:
        d = b - a
        return 1 / d


def true_prior_strain(x, a, b):
    m = 0.5 * (a + b)
    d = b - a

    if x < a:
        return 1 - m
    elif x > b:
        return -m
    else:
        return (-2 * x + (a**2 - b**2 + 2 * b)) / (2 * d)


def true_prior_solution(x, a, b):
    m = 0.5 * (a + b)
    d = b - a

    if x < a:
        return x * (1 - m)
    elif x > b:
        return (1 - x) * m
    else:
        return (-(x**2) + x * (a**2 - b**2 + 2 * b) - a**2) / (2 * d)


def true_prior_std(a, b):
    m = 0.5 * (a + b)
    d = b - a
    return np.sqrt(m * (1 - m) - d / 6)

In [ ]:
def fem_prior_source(a, b, *, mesh):
    nodes, elems = mesh
    n = len(nodes) - 1
    g = np.zeros(n + 1)

    start = int(a * n)
    end = int(b * n) + 1

    for inode in range(start, end + 1):
        if inode < n:
            curr_coord = nodes[inode][0]
            next_coord = nodes[inode + 1][0]

            if curr_coord < b and next_coord > a:
                lower = max(curr_coord, a)
                upper = min(next_coord, b)
                mid = 0.5 * (lower + upper)
                phi = 1 - (mid - curr_coord) / (next_coord - curr_coord)
                g[inode] += (upper - lower) / (b - a) * phi

        if inode > 0:
            prev_coord = nodes[inode - 1][0]
            curr_coord = nodes[inode][0]

            if prev_coord < b and curr_coord > a:
                lower = max(prev_coord, a)
                upper = min(curr_coord, b)
                mid = 0.5 * (lower + upper)
                phi = 1 - (curr_coord - mid) / (curr_coord - prev_coord)
                g[inode] += (upper - lower) / (b - a) * phi

    return g


def fem_prior_solution(a, b, *, mesh, K=None, g=None):
    nodes, elems = mesh

    if K is None:
        props = get_fem_props()
        jive = CJiveRunner(props, elems=elems)
        globdat = jive()
        K = globdat["matrix0"]

    if g is None:
        g = fem_prior_source(a, b, mesh=mesh)

    Kc = K[1:-1, 1:-1]
    gc = g[1:-1]
    ug = np.zeros_like(g)
    ug[1:-1] = spsolve(Kc, gc)
    return ug


def fem_prior_std(a, b, *, mesh, K=None):
    g = fem_prior_source(a, b, mesh=mesh)
    ug = fem_prior_solution(a, b, mesh=mesh, K=K, g=g)
    return np.sqrt(ug @ g)

In [ ]:
def true_posterior_std(a, b, *, obs_mesh, K_obs=None):
    sigma_prior = true_prior_std(a, b)
    sigma_downdate = fem_prior_std(a, b, mesh=obs_mesh, K=K_obs)
    return np.sqrt(sigma_prior**2 - sigma_downdate**2)


def true_posterior_mean(a, b, *, obs_mesh):
    obs_nodes, obs_elems = obs_mesh
    g = fem_prior_source(a, b, mesh=obs_mesh)
    props = get_fem_props()
    jive = CJiveRunner(props, elems=obs_elems)
    globdat = jive()
    K = globdat["matrix0"]
    f = globdat["extForce"]
    Kc = K[1:-1, 1:-1]
    fc = f[1:-1]
    uf = np.zeros_like(f)
    uf[1:-1] = spsolve(Kc, fc)
    return g @ uf


def fem_posterior_std(a, b, *, obs_mesh, ref_mesh, K_obs=None, K_ref=None):
    sigma_prior_h = fem_prior_std(a, b, mesh=ref_mesh, K=K_ref)
    sigma_downdate = fem_prior_std(a, b, mesh=obs_mesh, K=K_obs)
    return np.sqrt(sigma_prior_h**2 - sigma_downdate**2)

In [ ]:
f_l = np.zeros_like(domain)
eps_l = np.zeros_like(domain)
u_l = np.zeros_like(domain)

a = 0.49
b = 0.51

for i, x in enumerate(domain):
    f_l[i] = true_prior_source(x, a, b)
    eps_l[i] = true_prior_strain(x, a, b)
    u_l[i] = true_prior_solution(x, a, b)

In [ ]:
fig, axs = plt.subplots(ncols=3, figsize=(9, 3))
axs[0].plot(domain, f_l)
axs[1].plot(domain, eps_l)
axs[2].plot(domain, u_l)
plt.show()

In [ ]:
d = 0.01
ns = 2 ** np.arange(2, 18)
ms = np.linspace(0, 1, 15, endpoint=False)
ms += 0.5 * (ms[1] - ms[0])
prior_ref_W2_distances = np.zeros((len(ns), len(ms)))

for i, n in enumerate(ns):
    ref_mesh = mesh_interval_with_line2(n=n)
    ref_nodes, ref_elems = ref_mesh
    props = get_fem_props()
    jive = CJiveRunner(props, elems=ref_elems)
    globdat = jive()
    K = globdat["matrix0"]

    for j, m in enumerate(ms):
        a = m - 0.5 * d
        b = m + 0.5 * d
        sigma_prior = true_prior_std(a, b)
        sigma_prior_h = fem_prior_std(a, b, mesh=ref_mesh, K=K)
        prior_ref_W2_distances[i, j] = abs(sigma_prior - sigma_prior_h)

In [ ]:
fig, ax = plt.subplots()
ax.loglog(ns, prior_ref_W2_distances)
plt.show()

### 1D posterior convergence with reference mesh refinement

In [ ]:
obs_mesh = mesh_interval_with_line2(n=4)
obs_nodes, obs_elems = obs_mesh
props = get_fem_props()
jive = CJiveRunner(props, elems=obs_elems)
globdat = jive()
K_obs = globdat["matrix0"]

post_ref_W2_distances = np.zeros((len(ns), len(ms)))

for i, n in enumerate(ns):
    ref_mesh = mesh_interval_with_line2(n=n)
    ref_nodes, ref_elems = ref_mesh
    props = get_fem_props()
    jive = CJiveRunner(props, elems=ref_elems)
    globdat = jive()
    K_ref = globdat["matrix0"]

    for j, m in enumerate(ms):
        a = m - 0.5 * d
        b = m + 0.5 * d
        sigma_post = true_posterior_std(a, b, obs_mesh=obs_mesh, K_obs=K_obs)
        sigma_post_h = fem_posterior_std(
            a, b, obs_mesh=obs_mesh, ref_mesh=ref_mesh, K_obs=K_obs, K_ref=K_ref
        )
        post_ref_W2_distances[i, j] = abs(sigma_post - sigma_post_h)

In [ ]:
fig, ax = plt.subplots()
ax.loglog(ns, post_ref_W2_distances)
plt.show()

### 1D posterior convergence with observation mesh refinement

In [ ]:
post_obs_W2_distances = np.zeros((len(ns), len(ms)))
mu_trues = np.zeros((len(ns), len(ms)))
mu_posts = np.zeros((len(ns), len(ms)))

for i, n in enumerate(ns):
    obs_mesh = mesh_interval_with_line2(n=n)
    obs_nodes, obs_elems = obs_mesh
    props = get_fem_props()
    jive = CJiveRunner(props, elems=obs_elems)
    globdat = jive()
    K_obs = globdat["matrix0"]

    for j, m in enumerate(ms):
        a = m - 0.5 * d
        b = m + 0.5 * d
        mu_post = true_posterior_mean(a, b, obs_mesh=obs_mesh)
        mu_true = true_avg_solution(a, b)
        sigma_post = true_posterior_std(a, b, obs_mesh=obs_mesh, K_obs=K_obs)
        post_obs_W2_distances[i, j] = np.sqrt((mu_true - mu_post) ** 2 + sigma_post**2)

In [ ]:
fig, ax = plt.subplots()
ax.loglog(ns, post_obs_W2_distances)
plt.show()